In [116]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
import frac_diff as fd
CONFIG_PATH = "../config.yaml"

In [117]:
gold = pl.read_parquet("../data/raw/Gold_1d.parquet")
nifty = pl.read_parquet("../data/raw/Nifty50_1d.parquet")
usdinr = pl.read_parquet("../data/raw/USDINR_1d.parquet")

In [118]:
nifty = nifty.rename({"('Close', '^NSEI')":"Close","('High', '^NSEI')":"High","('Low', '^NSEI')":"Low","('Open', '^NSEI')":"Open","('Volume', '^NSEI')":"Volume"})
gold = gold.rename({"('Close', 'GOLDBEES.NS')":"Close","('High', 'GOLDBEES.NS')":"High","('Low', 'GOLDBEES.NS')":"Low","('Open', 'GOLDBEES.NS')":"Open","('Volume', 'GOLDBEES.NS')":"Volume"})
usdinr = usdinr.rename({"('Close', 'USDINR=X')":"Close","('High', 'USDINR=X')":"High","('Low', 'USDINR=X')":"Low","('Open', 'USDINR=X')":"Open","('Volume', 'USDINR=X')":"Volume"})


In [119]:
print(gold.null_count().sum())
print(usdinr.null_count().sum())
print(nifty.null_count().sum())

shape: (1, 6)
┌───────┬──────┬─────┬──────┬────────┬──────┐
│ Close ┆ High ┆ Low ┆ Open ┆ Volume ┆ Date │
│ ---   ┆ ---  ┆ --- ┆ ---  ┆ ---    ┆ ---  │
│ u32   ┆ u32  ┆ u32 ┆ u32  ┆ u32    ┆ u32  │
╞═══════╪══════╪═════╪══════╪════════╪══════╡
│ 0     ┆ 0    ┆ 0   ┆ 0    ┆ 0      ┆ 0    │
└───────┴──────┴─────┴──────┴────────┴──────┘
shape: (1, 6)
┌───────┬──────┬─────┬──────┬────────┬──────┐
│ Close ┆ High ┆ Low ┆ Open ┆ Volume ┆ Date │
│ ---   ┆ ---  ┆ --- ┆ ---  ┆ ---    ┆ ---  │
│ u32   ┆ u32  ┆ u32 ┆ u32  ┆ u32    ┆ u32  │
╞═══════╪══════╪═════╪══════╪════════╪══════╡
│ 0     ┆ 0    ┆ 0   ┆ 0    ┆ 0      ┆ 0    │
└───────┴──────┴─────┴──────┴────────┴──────┘
shape: (1, 6)
┌───────┬──────┬─────┬──────┬────────┬──────┐
│ Close ┆ High ┆ Low ┆ Open ┆ Volume ┆ Date │
│ ---   ┆ ---  ┆ --- ┆ ---  ┆ ---    ┆ ---  │
│ u32   ┆ u32  ┆ u32 ┆ u32  ┆ u32    ┆ u32  │
╞═══════╪══════╪═════╪══════╪════════╪══════╡
│ 0     ┆ 0    ┆ 0   ┆ 0    ┆ 0      ┆ 0    │
└───────┴──────┴─────┴──────┴────────┴

In [120]:
print(nifty.select(pl.col("Date").is_duplicated().sum()))
print(gold.select(pl.col("Date").is_duplicated().sum()))
print(usdinr.select(pl.col("Date").is_duplicated().sum()))

shape: (1, 1)
┌──────┐
│ Date │
│ ---  │
│ u32  │
╞══════╡
│ 0    │
└──────┘
shape: (1, 1)
┌──────┐
│ Date │
│ ---  │
│ u32  │
╞══════╡
│ 0    │
└──────┘
shape: (1, 1)
┌──────┐
│ Date │
│ ---  │
│ u32  │
╞══════╡
│ 0    │
└──────┘


In [121]:
print((nifty['Close'] <= 0).sum())
print((gold['Close'] <= 0).sum())
print((usdinr['Close'] <= 0).sum())

0
0
0


In [122]:
usdinr.head()

Close,High,Low,Open,Volume,Date
f64,f64,f64,f64,i64,datetime[ns]
53.007999,53.330002,53.099998,53.099998,0,2012-01-02 00:00:00
53.298,53.298,53.049999,53.298,0,2012-01-03 00:00:00
53.049999,53.209999,52.849998,53.209999,0,2012-01-04 00:00:00
52.849998,53.040001,52.608002,53.040001,0,2012-01-05 00:00:00
52.759998,52.869999,52.599998,52.759998,0,2012-01-06 00:00:00


In [123]:
bad_dates = [
    pl.datetime(2019, 12, 19),
    pl.datetime(2019, 12, 20),
]
for date in bad_dates:
    nifty = nifty.filter(pl.col("Date") != date)
    gold = gold.filter(pl.col("Date") != date)
    usdinr = usdinr.filter(pl.col("Date") != date)

In [124]:
nifty_dates = set(nifty["Date"].to_list())
gold_dates  = set(gold["Date"].to_list())
usd_dates   = set(usdinr["Date"].to_list())

common = nifty_dates & gold_dates & usd_dates
print(f"Common trading days: {len(common)}")
print(f"Nifty-only days:     {len(nifty_dates - common)}")
print(f"Gold-only days:      {len(gold_dates - common)}")
print(f"USDINR-only days:    {len(usd_dates - common)}")

Common trading days: 3446
Nifty-only days:     9
Gold-only days:      23
USDINR-only days:    218


In [125]:
common_dates = nifty.select("Date") \
    .join(gold.select("Date"), on="Date", how="inner") \
    .join(usdinr.select("Date"), on="Date", how="inner") \
    .unique()

In [126]:
nifty = nifty.join(common_dates, on="Date", how="inner")
gold = gold.join(common_dates, on="Date", how="inner")
usdinr = usdinr.join(common_dates, on="Date", how="inner")

In [127]:
print(nifty['Date'].to_list() == gold['Date'].to_list() == usdinr['Date'].to_list())

True


In [128]:
usdinr.head()

Close,High,Low,Open,Volume,Date
f64,f64,f64,f64,i64,datetime[ns]
53.298,53.298,53.049999,53.298,0,2012-01-03 00:00:00
53.049999,53.209999,52.849998,53.209999,0,2012-01-04 00:00:00
52.849998,53.040001,52.608002,53.040001,0,2012-01-05 00:00:00
52.759998,52.869999,52.599998,52.759998,0,2012-01-06 00:00:00
52.673,52.845001,52.369999,52.810001,0,2012-01-09 00:00:00


In [129]:
usdinr = usdinr.drop("Volume")
nifty = nifty.drop("Volume")
gold = gold.drop("Volume")

In [130]:
usdinr.head()

Close,High,Low,Open,Date
f64,f64,f64,f64,datetime[ns]
53.298,53.298,53.049999,53.298,2012-01-03 00:00:00
53.049999,53.209999,52.849998,53.209999,2012-01-04 00:00:00
52.849998,53.040001,52.608002,53.040001,2012-01-05 00:00:00
52.759998,52.869999,52.599998,52.759998,2012-01-06 00:00:00
52.673,52.845001,52.369999,52.810001,2012-01-09 00:00:00


In [131]:
nifty.write_parquet("../data/processed//nifty_cleaned.parquet")
gold.write_parquet("../data/processed/gold_cleaned.parquet")
usdinr.write_parquet("../data/processed/usdinr_cleaned.parquet")